In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
# from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.base import clone

import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import os
from tqdm import tqdm

from prepare_data import prepare_data

import random

In [255]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [256]:
data = pd.read_csv('data/train.csv')
data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [257]:
y = np.log1p(data['SalePrice'])
X = prepare_data(data.drop('SalePrice',axis=1))

In [258]:
X = X[X['GrLivArea'] < 4500]
#синхронизируем данные
y = y[X.index]

In [259]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(0.2/0.9), random_state=SEED)

In [260]:
numeric_colums = X_train.select_dtypes(include=['number']).columns
categorical_colums = X_train.select_dtypes(include=['object', 'string']).columns

In [261]:
low_card_cols = [
    col for col in categorical_colums
    if X_train[col].nunique() < 5
]

high_card_cols = [
    col for col in categorical_colums
    if X_train[col].nunique() >= 5
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    # ('variance_selector', VarianceThreshold(threshold=0.1))
])

categorical_imputer = SimpleImputer(strategy='constant', fill_value='missing')  


preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', Pipeline([
            ('imputer', categorical_imputer),
            ('encoder', OneHotEncoder(
            drop='first',
            handle_unknown='ignore',
            sparse_output=False
        ))
        ]), low_card_cols),

        ('target', Pipeline([
            ('imputer', categorical_imputer),
            ('target',  TargetEncoder(
            target_type='continuous',
            random_state=SEED
        ))]), high_card_cols),

        ('numeric', numeric_transformer, numeric_colums)
    ],
    verbose_feature_names_out=False
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...), ('target', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_nam

In [262]:
preview_preprocessor = clone(preprocessor)
preview_preprocessor.set_output(transform='pandas')

X_encoded_preview = preview_preprocessor.fit_transform(X, y)
X_encoded_preview.head()

,Street_Pave,Alley_Grvl,Alley_Pave,LotShape_IR2,LotShape_IR3,LotShape_Reg,LandContour_HLS,LandContour_Low,LandContour_Lvl,Utilities_NoSeWa,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,TotalSF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,-0.750831,0.225982,-0.359603,-0.11642,-0.270407,-0.063709,-0.087748,-1.601578,0.138375,0.011436
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.627328,-0.708304,-0.359603,-0.11642,-0.270407,-0.063709,-0.087748,-0.490155,-0.614427,-0.042838
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.750831,-0.065025,-0.359603,-0.11642,-0.270407,-0.063709,-0.087748,0.991743,0.138375,0.192351
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.750831,-0.172238,4.089589,-0.11642,-0.270407,-0.063709,-0.087748,-1.601578,-1.367230,-0.108743
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.781406,0.578253,-0.359603,-0.11642,-0.270407,-0.063709,-0.087748,2.103167,0.138375,1.015514


In [263]:
generator = torch.Generator().manual_seed(SEED)

# Применяем препроцессор
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Преобразуем данные в тензоры PyTorch
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Создаем TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Создаем DataLoader'ы
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True, generator=generator)
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

c:\Users\angelika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [8, 9, 14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\angelika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [264]:

class MyNN(nn.Module):
    def __init__(self, input, output):
        super().__init__()
        self.layer_1 = nn.Linear(input,128)
        self.batch_norm_1 = nn.BatchNorm1d(128)
        self.act_1 = nn.LeakyReLU()
        self.dropout_1 = nn.Dropout(0.2)
        self.layer_2 = nn.Linear(128,64)
        self.batch_norm_2 = nn.BatchNorm1d(64)
        self.act_2 = nn.LeakyReLU()
        self.dropout_2 = nn.Dropout(0.2)
        self.layer_3 = nn.Linear(64,32)
        self.batch_norm_3 = nn.BatchNorm1d(32)
        self.act_3 = nn.ReLU()
        self.dropout_3 = nn.Dropout(0.2)
        self.layer_4 = nn.Linear(32,output)

    def forward(self,X):
        X = self.layer_1(X)
        X = self.batch_norm_1(X)
        X = self.act_1(X)
        X = self.dropout_1(X)
        X = self.layer_2(X)
        X = self.batch_norm_2(X)
        X = self.act_2(X)
        X = self.dropout_2(X)
        X = self.layer_3(X)
        X = self.batch_norm_3(X)
        X = self.act_3(X)
        X = self.dropout_3(X)
        out = self.layer_4(X)
        return out

In [265]:
model = MyNN(X_train_processed.shape[1], 1)


In [266]:
loss_reg = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

In [267]:
from torch.optim import lr_scheduler

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,  #оптимизатор
                                                          mode = 'min',  #'max' или 'min' - следим чтобы отслеживаемый параметр увеличивался()
                                                          factor = 0.1,  #коэфициэнт, на который будет умножен lr
                                                          patience = 10, #кол-во эпох без улучшения отслеживаемого пораметров
                                                          eps = 1e-8,
                                                          )

In [ ]:


EPOCHS = 500
train_loss = []
val_loss = []
lr_list = []
best_loss = None
best_model = 0
count = 0

for epoch in range(EPOCHS):
    model.train()
    train_loop = tqdm(train_loader,leave=False)
    running_train_loss = []
    for x,targets in train_loop:

        pred = model(x)
        loss = torch.sqrt(loss_reg(pred, targets))

        opt.zero_grad()
        loss.backward()

        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss)/len(running_train_loss)

        train_loop.set_description(f'Epoch[{epoch+1}/{EPOCHS}], train_loss = {mean_train_loss:.4f}')

    train_loss.append(mean_train_loss)

    model.eval()
    with torch.no_grad():
        running_val_loss = []
        for x, targets in val_loader:
            pred = model(x)
            loss = torch.sqrt(loss_reg(pred,targets))

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss)/len(running_val_loss)

        val_loss.append(mean_val_loss)

    lr_scheduler.step(mean_val_loss)

    lr = lr_scheduler._last_lr[0]
    lr_list.append(lr)
    print((f'Epoch [{epoch+1}/{EPOCHS}], train_loss = {mean_train_loss:.4f},  val_loss = {mean_val_loss:.4f}, lr = {lr:.4f}'))

    if best_loss is None:
      best_loss = mean_val_loss

    if mean_val_loss < best_loss:
      best_loss = mean_val_loss
      count = 0
      checkpoint = {
          'state_model': model.state_dict(),
          'state_opt' : opt.state_dict(),
          'state_lr_scheduler': lr_scheduler.state_dict(),
          'loss' : {
              'train_loss':train_loss,
              'val_loss': val_loss,
              'best_loss': best_loss,
          },
          'metric':{
              'train_loss' : train_loss,
              'val_loss' : val_loss,
          },
          'lr' : lr_list,
          'epoch' : {
              'EPOCHS' : EPOCHS,
              'save_epoch': epoch
          }
      }

      if best_model == 0:
        torch.save(model.state_dict(), f'best_model/model_state_dict_epoch_{epoch+1}.pt')
        best_model = epoch+1

      else:
        os.remove(f'best_model/model_state_dict_epoch_{best_model}.pt')
        torch.save(model.state_dict(), f'best_model/model_state_dict_epoch_{epoch+1}.pt')
        best_model = epoch+1

      print(f'На эпохе - {epoch+1}, сохранена модель со значением функции потерь на валидации - {best_loss:.4f}', end='\n\n') 

    if count >= 150:
      print(f'\033[31Обучение остановлено на {epoch+1} эпохе.\033[0m')
      break
    count += 1

  0%|          | 0/32 [00:00<?, ?it/s]

Epoch [1/500], train_loss = 8.9704,  val_loss = 2.8008, lr = 0.0100


Epoch [2/500], train_loss = 2.3945,  val_loss = 2.6125, lr = 0.0100
На эпохе - 2, сохранена модель со значением функции потерь на валидации - 2.6125



Epoch [3/500], train_loss = 1.8404,  val_loss = 1.0275, lr = 0.0100
На эпохе - 3, сохранена модель со значением функции потерь на валидации - 1.0275



Epoch [4/500], train_loss = 1.6725,  val_loss = 0.8676, lr = 0.0100
На эпохе - 4, сохранена модель со значением функции потерь на валидации - 0.8676



...

Epoch [285/500], train_loss = 0.5141,  val_loss = 0.1573, lr = 0.0000


Epoch [286/500], train_loss = 0.5033,  val_loss = 0.1595, lr = 0.0000


Epoch [287/500], train_loss = 0.5301,  val_loss = 0.1557, lr = 0.0000


Epoch [288/500], train_loss = 0.5393,  val_loss = 0.1772, lr = 0.0000


Epoch [289/500], train_loss = 0.5336,  val_loss = 0.1670, lr = 0.0000
[31Обучение остановлено на 289 эпохе.


In [269]:
model.load_state_dict(torch.load(f'best_model/model_state_dict_epoch_{best_model}.pt'))

model.eval()
with torch.no_grad():
    running_test_loss = []
    for x, targets in test_loader:
        pred = model(x)
        loss = torch.sqrt(loss_reg(pred,targets))
        running_test_loss.append(loss.item())

    mean_test_loss = sum(running_test_loss)/len(running_test_loss)
    

    print(f'Финальный loss = {mean_test_loss:.4f}')

Финальный loss = 0.1319


In [270]:
# 1. Загружаем веса лучшей модели (используем best_model из вашего цикла)
model.load_state_dict(torch.load(f'best_model/model_state_dict_epoch_{best_model}.pt'))
model.eval()

# 2. Загружаем тестовый набор данных Kaggle
test_data = pd.read_csv('data/test.csv')
test_ids = test_data['Id']

# 3. Предобработка тестовых данных
# Используем ту же функцию prepare_data и тот же preprocessor (только transform!)
X_test_final = prepare_data(test_data)
X_test_processed_final = preprocessor.transform(X_test_final)

# Конвертируем в тензор
X_test_tensor_final = torch.tensor(X_test_processed_final, dtype=torch.float32)

print("Данные готовы к предсказанию.")

c:\Users\angelika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [4, 8, 11] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Данные готовы к предсказанию.


In [271]:
with torch.no_grad():
    # Получаем предсказания
    log_preds = model(X_test_tensor_final).numpy().flatten()

# Переводим из логарифмического масштаба обратно
final_preds = np.expm1(log_preds)

# Создаем DataFrame для отправки
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_preds
})

# Сохраняем в CSV
submission.to_csv('data/nn/submission.csv', index=False)

print("Файл submission.csv успешно создан!")
display(submission.head())

Файл submission.csv успешно создан!


,Id,SalePrice
0,1461,118698.796875
1,1462,204890.937500
2,1463,167214.484375
3,1464,178270.421875
4,1465,178570.578125
